# exp_009: θ0 vs θ1 のCOCO予測 比較可視化ノートブック

## 目的
事前学習済みモデル **θ0** と、aerialドメインで学習した **θ1** を、**同一のCOCO画像・同一プロンプト・同一閾値**で
推論し、検出結果を**左右に並べて**比較する。指標は計算せず、忘却が
**(1) boxごと消失 / (2) スコア低下 / (3) ラベル変化** のどれとして現れるかを**目視**で当たりをつけるためのもの。

## モデル
- **θ0** = 事前学習済み MM-Grounding DINO（公開重み）
- **θ1** = aerial 単独学習（exp_006 の逐次の起点に使ったモデル＝exp_005 aerial）

## 画像の選び方（2モード）
- `SELECTION_MODE = 'random'` : **seed固定**でランダムに N 枚（オブジェクト数 >= MIN_OBJECTS で絞り込み可）
- `SELECTION_MODE = 'manual'` : `MANUAL_IMAGES` にファイル名を列挙して指定

## 使い方
1. **セル1（設定）** でパス・モード・枚数などを指定
2. 上から順に実行
3. `OUT_DIR` に保存された比較画像を見て、(1)/(2)/(3) のどれが支配的かを `outputs/notes.md` に記録


## セル1：設定

In [18]:
# このノートブックは config 等を「リポジトリのルートからの相対パス」で指定する。
# Jupyter の作業ディレクトリが exp_009/ などでも動くよう、最初にルートへ移動する。
import os
REPO_ROOT = '/workspace/kouyou/mmdetection'
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

# === モデル（θ0: 事前学習済み / θ1: aerial 学習後）===
CONFIG_THETA0 = 'configs/mm_grounding_dino/grounding_dino_swin-t_pretrain_obj365.py'
CKPT_THETA0   = 'grounding_dino_swin-t_pretrain_obj365_goldg_grit9m_v3det_20231204_095047-b448804b.pth'

CONFIG_THETA1 = 'configs/mm_grounding_dino/grounding_dino_swin-t_finetune_8xb4_20e_aerial.py'
CKPT_THETA1   = 'experiments/exp_005/aerial_work_dir/best_coco_bbox_mAP_epoch_17.pth'

LABEL_THETA0 = 'theta0 (pretrained)'
LABEL_THETA1 = 'theta1 (aerial)'

# === COCO val データ ===
COCO_IMG_DIR = '/workspace/kouyou/datasets/coco2017/val2017'
COCO_ANN     = '/workspace/kouyou/datasets/coco2017/annotations/instances_val2017.json'

# === 出力先（exp_009 配下）===
OUT_DIR = 'experiments/exp_009/outputs/figures'

# === 推論設定 ===
DEVICE       = 'cuda:0'
SCORE_THR    = 0.5      # 可視化のスコア閾値（θ0/θ1 で共通）
CHUNKED_SIZE = -1       # カテゴリ分割。COCO80クラスは256トークンに収まるため -1(無効)でよい

# === 画像選択モード ===
SELECTION_MODE = 'random'   # 'random' または 'manual'

# -- random モードの設定 --
SEED        = 0    # seed固定で再現可能
N_IMAGES    = 50   # 選ぶ枚数
MIN_OBJECTS = 3    # この数以上のアノテーションを持つ画像だけを対象にする（0で絞り込み無し）

# -- manual モードの設定 --
# val2017 のファイル名で指定（例: '000000000139.jpg'）
MANUAL_IMAGES = [
    # '000000000139.jpg',
    # '000000000285.jpg',
]


cwd = /workspace/kouyou/mmdetection


## セル2：COCOクラス名 → テキストプロンプト
COCO 80 クラス名を取得し、Grounding DINO 用のピリオド区切りプロンプトを作る。

In [19]:
from mmdet.evaluation import get_classes

coco_classes = list(get_classes('coco'))
texts_prompt = ' . '.join(coco_classes) + ' .'

print(f'COCOクラス数: {len(coco_classes)}')
print('プロンプト先頭:', texts_prompt[:120], '...')


COCOクラス数: 80
プロンプト先頭: person . bicycle . car . motorcycle . airplane . bus . train . truck . boat . traffic_light . fire_hydrant . stop_sign . ...


## セル3：対象画像の選択（random / manual）
`SELECTION_MODE` に応じて画像パスのリスト `selected_paths` を作る。

In [20]:
import os, json, random

with open(COCO_ANN) as f:
    coco = json.load(f)

id_to_file = {img['id']: img['file_name'] for img in coco['images']}

if SELECTION_MODE == 'random':
    # 画像ごとのアノテーション数を数える
    from collections import Counter
    obj_count = Counter(ann['image_id'] for ann in coco['annotations'])
    # MIN_OBJECTS 以上のオブジェクトを持つ画像に絞る
    candidate_ids = [iid for iid, fn in id_to_file.items()
                     if obj_count.get(iid, 0) >= MIN_OBJECTS]
    candidate_ids.sort()  # 決定的に並べてから seed固定でサンプリング
    rng = random.Random(SEED)
    chosen_ids = rng.sample(candidate_ids, min(N_IMAGES, len(candidate_ids)))
    selected_files = [id_to_file[i] for i in chosen_ids]
    print(f'random選択: 候補{len(candidate_ids)}枚(>= {MIN_OBJECTS}obj) から seed={SEED} で {len(selected_files)}枚')

elif SELECTION_MODE == 'manual':
    selected_files = list(MANUAL_IMAGES)
    print(f'manual選択: {len(selected_files)}枚')

else:
    raise ValueError(f"SELECTION_MODE は 'random' か 'manual': {SELECTION_MODE}")

selected_paths = [os.path.join(COCO_IMG_DIR, fn) for fn in selected_files]
# 存在チェック
missing = [p for p in selected_paths if not os.path.exists(p)]
assert not missing, f'存在しない画像: {missing}'
print('\n'.join(selected_files))


random選択: 候補3434枚(>= 3obj) から seed=0 で 50枚
000000269113.jpg
000000523100.jpg
000000292446.jpg
000000025096.jpg
000000178469.jpg
000000356248.jpg
000000337498.jpg
000000283318.jpg
000000543043.jpg
000000576031.jpg
000000210520.jpg
000000330396.jpg
000000248112.jpg
000000406611.jpg
000000151657.jpg
000000350405.jpg
000000092939.jpg
000000192904.jpg
000000093261.jpg
000000521717.jpg
000000061471.jpg
000000430377.jpg
000000554579.jpg
000000172648.jpg
000000369812.jpg
000000485480.jpg
000000562843.jpg
000000419312.jpg
000000098716.jpg
000000215072.jpg
000000064523.jpg
000000504415.jpg
000000047769.jpg
000000471869.jpg
000000228942.jpg
000000327769.jpg
000000388258.jpg
000000065798.jpg
000000245026.jpg
000000300913.jpg
000000220764.jpg
000000425925.jpg
000000446206.jpg
000000142238.jpg
000000382743.jpg
000000330790.jpg
000000306437.jpg
000000363207.jpg
000000179265.jpg
000000039769.jpg


## セル4：モデル読み込み（θ0, θ1）
2つの `DetInferencer` を構築する。

In [21]:
from mmdet.apis import DetInferencer

def build(config, ckpt):
    inf = DetInferencer(model=config, weights=ckpt, device=DEVICE, palette='none')
    inf.model.test_cfg.chunked_size = CHUNKED_SIZE
    return inf

inf0 = build(CONFIG_THETA0, CKPT_THETA0)
inf1 = build(CONFIG_THETA1, CKPT_THETA1)
print('θ0, θ1 を読み込みました。')


Loads checkpoint by local backend from path: grounding_dino_swin-t_pretrain_obj365_goldg_grit9m_v3det_20231204_095047-b448804b.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: language_model.language_backbone.body.model.embeddings.position_ids

Loads checkpoint by local backend from path: experiments/exp_005/aerial_work_dir/best_coco_bbox_mAP_epoch_17.pth


/opt/conda/lib/python3.10/site-packages/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.LocalVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '


θ0, θ1 を読み込みました。


## セル5：推論＋左右並べて可視化・保存
各画像について θ0（左）/ θ1（右）の検出を mmdet の描画で並べ、`OUT_DIR` に保存する。

In [22]:
import os
import numpy as np
from PIL import Image, ImageDraw, ImageFont

os.makedirs(OUT_DIR, exist_ok=True)

def load_font(size):
    try:
        return ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', size)
    except Exception:
        return ImageFont.load_default()
HEADER_FONT = load_font(22)

def vis_one(inf, img_path):
    """1枚を推論し、mmdetが描画したRGB画像(ndarray)を返す。"""
    res = inf(
        inputs=img_path,
        texts=texts_prompt,
        custom_entities=True,
        pred_score_thr=SCORE_THR,
        return_datasamples=False,
        no_save_vis=True,
        no_save_pred=True,
        return_vis=True,
    )
    return res['visualization'][0]  # RGB ndarray

def add_header(pil_img, text, bar_h=34):
    """画像上部にラベル帯を付ける。"""
    w, h = pil_img.size
    canvas = Image.new('RGB', (w, h + bar_h), (0, 0, 0))
    canvas.paste(pil_img, (0, bar_h))
    d = ImageDraw.Draw(canvas)
    d.text((8, 6), text, fill=(255, 255, 255), font=HEADER_FONT)
    return canvas

def compare_and_save(img_path):
    fn = os.path.basename(img_path)
    v0 = add_header(Image.fromarray(vis_one(inf0, img_path)), LABEL_THETA0)
    v1 = add_header(Image.fromarray(vis_one(inf1, img_path)), LABEL_THETA1)
    # 高さを揃えて左右連結
    h = max(v0.height, v1.height)
    def fit(im):
        if im.height != h:
            im = im.resize((int(im.width * h / im.height), h))
        return im
    v0, v1 = fit(v0), fit(v1)
    gap = 8
    out = Image.new('RGB', (v0.width + gap + v1.width, h), (255, 255, 255))
    out.paste(v0, (0, 0))
    out.paste(v1, (v0.width + gap, 0))
    save_path = os.path.join(OUT_DIR, f'cmp_{fn.replace(".jpg","")}.png')
    out.save(save_path)
    return save_path

saved = []
for p in selected_paths:
    sp = compare_and_save(p)
    saved.append(sp)
    print('saved:', sp)

print(f'\n完了: {len(saved)}枚を {OUT_DIR} に保存しました（左=θ0 / 右=θ1）。')


/opt/conda/lib/python3.10/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/conda/lib/python3.10/site-packages/mmcv/cnn/bricks/transformer.py:524: UserWarning: position encoding of key 
ismissing in MultiheadAttention.
  warnings.warn(f'position encoding of key is'

saved: experiments/exp_009/outputs/figures/cmp_000000269113.png


saved: experiments/exp_009/outputs/figures/cmp_000000523100.png


saved: experiments/exp_009/outputs/figures/cmp_000000292446.png


...

saved: experiments/exp_009/outputs/figures/cmp_000000330790.png


saved: experiments/exp_009/outputs/figures/cmp_000000306437.png


saved: experiments/exp_009/outputs/figures/cmp_000000363207.png


saved: experiments/exp_009/outputs/figures/cmp_000000179265.png


saved: experiments/exp_009/outputs/figures/cmp_000000039769.png

完了: 50枚を experiments/exp_009/outputs/figures に保存しました（左=θ0 / 右=θ1）。


## セル6：観察メモのテンプレート
保存した比較画像を見て、各画像で **(1) boxごと消失 / (2) スコア低下 / (3) ラベル変化** のどれが起きているかを記録する。
所見は `experiments/exp_009/outputs/notes.md` にまとめる。

In [12]:
# 観察を素早くメモするための雛形（必要なら編集して使う）
template = """# exp_009 観察メモ（θ0 vs θ1, COCO {n}枚）

## 各画像の所見（(1)box消失 / (2)スコア低下 / (3)ラベル変化）
| 画像 | 支配的な類型 | メモ |
|---|---|---|
"""
for p in selected_paths:
    template += f"| {os.path.basename(p)} |  |  |\n"

template += """
## 全体の傾向
- 支配的だったのは ( 1 / 2 / 3 ) のどれか：
- 次に定量すべきと考えたこと：
"""
print(template.format(n=len(selected_paths)))


# exp_009 観察メモ（θ0 vs θ1, COCO 50枚）

## 各画像の所見（(1)box消失 / (2)スコア低下 / (3)ラベル変化）
| 画像 | 支配的な類型 | メモ |
|---|---|---|
| 000000269113.jpg |  |  |
| 000000523100.jpg |  |  |
| 000000292446.jpg |  |  |
| 000000025096.jpg |  |  |
| 000000178469.jpg |  |  |
| 000000356248.jpg |  |  |
| 000000337498.jpg |  |  |
| 000000283318.jpg |  |  |
| 000000543043.jpg |  |  |
| 000000576031.jpg |  |  |
| 000000210520.jpg |  |  |
| 000000330396.jpg |  |  |
| 000000248112.jpg |  |  |
| 000000406611.jpg |  |  |
| 000000151657.jpg |  |  |
| 000000350405.jpg |  |  |
| 000000092939.jpg |  |  |
| 000000192904.jpg |  |  |
| 000000093261.jpg |  |  |
| 000000521717.jpg |  |  |
| 000000061471.jpg |  |  |
| 000000430377.jpg |  |  |
| 000000554579.jpg |  |  |
| 000000172648.jpg |  |  |
| 000000369812.jpg |  |  |
| 000000485480.jpg |  |  |
| 000000562843.jpg |  |  |
| 000000419312.jpg |  |  |
| 000000098716.jpg |  |  |
| 000000215072.jpg |  |  |
| 000000064523.jpg |  |  |
| 000000504415.jpg |  |  |
| 000000047769.jpg |  |